# Target Agent: Agent 2


### Agent Responsibility
> *"calulactes macros and use ollama llm when needed to map values and to explain the output of macros"*


In [1]:
from dataclasses import dataclass, field
from typing import Optional
import re

In [2]:
VALID_SEXES = {"male", "female"}

VALID_ACTIVITY_LEVELS = {"sedentary", "light", "moderate", "active"}

VALID_GOALS = {"weight_loss", "maintenance", "muscle_gain"}

VALID_DIET_TYPES = {"balanced", "vegetarian", "vegan", "keto", "low_carb"}

## Normalization Maps


In [3]:
ACTIVITY_NORM_MAP = {
    # sedentary
    "none": "sedentary",
    "no exercise": "sedentary",
    "desk job": "sedentary",
    "inactive": "sedentary",
    "couch": "sedentary",
    # light
    "light": "light",
    "walks": "light",
    "walking": "light",
    "1x week": "light",
    "1-2x week": "light",
    "light exercise": "light",
    # moderate
    "moderate": "moderate",
    "gym 3x": "moderate",
    "gym 3x/week": "moderate",
    "3x week": "moderate",
    "3-4x week": "moderate",
    "sometimes": "moderate",
    "regular": "moderate",
    # active
    "active": "active",
    "very active": "active",
    "daily": "active",
    "5x week": "active",
    "5-6x week": "active",
    "athlete": "active",
    "intense": "active",
}

GOAL_NORM_MAP = {
    # weight_loss
    "lose weight": "weight_loss",
    "lose fat": "weight_loss",
    "cut": "weight_loss",
    "cutting": "weight_loss",
    "slim down": "weight_loss",
    "fat loss": "weight_loss",
    "shred": "weight_loss",
    "get shredded": "weight_loss",
    "calorie deficit": "weight_loss",
    # maintenance
    "maintain": "maintenance",
    "maintain weight": "maintenance",
    "stay same": "maintenance",
    "keep weight": "maintenance",
    "healthy": "maintenance",
    # muscle_gain
    "gain muscle": "muscle_gain",
    "build muscle": "muscle_gain",
    "bulk": "muscle_gain",
    "bulking": "muscle_gain",
    "gain mass": "muscle_gain",
    "get bigger": "muscle_gain",
    "hypertrophy": "muscle_gain",
}

DIET_NORM_MAP = {
    "balanced": "balanced",
    "normal": "balanced",
    "everything": "balanced",
    "omnivore": "balanced",
    "vegetarian": "vegetarian",
    "veg": "vegetarian",
    "no meat": "vegetarian",
    "vegan": "vegan",
    "plant based": "vegan",
    "plant-based": "vegan",
    "keto": "keto",
    "ketogenic": "keto",
    "low carb": "low_carb",
    "low-carb": "low_carb",
    "low carbohydrate": "low_carb",
}

In [4]:
FIELD_DEFAULTS = {
    "medical_conditions": [],
    "diet_type": "balanced",
}

In [5]:
@dataclass
class UserProfile:

    age: int
    sex: str
    height_cm: float
    weight_kg: float
    activity_level: str
    goal: str
    medical_conditions: list = field(default_factory=list)
    diet_type: str = "balanced"


@dataclass
class ValidationResult:
    """
    Output of validate_input().

    valid         → False if any hard error exists
    profile       → UserProfile if valid, else None
    errors        → blocking issues (stop processing)
    warnings      → non-blocking flags (continue with caution)
    normalizations → log of what was auto-corrected
    """
    valid: bool
    profile: Optional[UserProfile]
    errors: list
    warnings: list
    normalizations: list

In [6]:
def _normalize_string(value: str) -> str:
    """Lowercase, strip, and collapse internal whitespace."""
    return re.sub(r"\s+", " ", str(value).strip().lower())


def _lookup_norm_map(value: str, norm_map: dict) -> Optional[str]:
    """Try exact match in a normalization map after cleaning."""
    return norm_map.get(_normalize_string(value))


def normalize_activity(raw: str) -> tuple:
    """
    Returns (canonical_value, was_already_canonical).
    Returns (None, False) if unrecognised → triggers LLM fallback.
    """
    cleaned = _normalize_string(raw)
    if cleaned in VALID_ACTIVITY_LEVELS:
        return cleaned, True
    mapped = _lookup_norm_map(cleaned, ACTIVITY_NORM_MAP)
    return mapped, False


def normalize_goal(raw: str) -> tuple:
    """Same pattern as normalize_activity."""
    cleaned = _normalize_string(raw)
    if cleaned in VALID_GOALS:
        return cleaned, True
    mapped = _lookup_norm_map(cleaned, GOAL_NORM_MAP)
    return mapped, False


def normalize_diet(raw: str) -> tuple:
    """Same pattern as normalize_activity."""
    cleaned = _normalize_string(raw)
    if cleaned in VALID_DIET_TYPES:
        return cleaned, True
    mapped = _lookup_norm_map(cleaned, DIET_NORM_MAP)
    return mapped, False

In [17]:
def validate_input(raw_input: dict) -> ValidationResult:

    errors         = []
    warnings       = []
    normalizations = []


    data = {**FIELD_DEFAULTS, **raw_input}

    # Check required keys exist
    REQUIRED = ["age", "sex", "height_cm", "weight_kg", "activity_level", "goal"]
    for key in REQUIRED:
        if key not in data or data[key] is None or str(data[key]).strip() == "":
            errors.append(f"Missing required field: '{key}'")

    if errors:
        return ValidationResult(False, None, errors, warnings, normalizations)


    # age
    try:
        age = int(data["age"])
        if not (1 <= age <= 120):
            errors.append(f"'age' out of plausible range: {age}")
    except (ValueError, TypeError):
        errors.append(f"'age' must be an integer, got: {data['age']!r}")
        age = None

    # sex
    sex_raw = _normalize_string(str(data["sex"]))
    if sex_raw not in VALID_SEXES:
        errors.append(f"'sex' must be 'male' or 'female', got: {data['sex']!r}")
        sex = None
    else:
        sex = sex_raw

    # height_cm
    try:
        height_cm = float(data["height_cm"])
        if not (50 <= height_cm <= 250):
            errors.append(f"'height_cm' out of plausible range: {height_cm}")
    except (ValueError, TypeError):
        errors.append(f"'height_cm' must be numeric, got: {data['height_cm']!r}")
        height_cm = None

    # weight_kg
    try:
        weight_kg = float(data["weight_kg"])
        if not (20 <= weight_kg <= 300):
            errors.append(f"'weight_kg' out of plausible range: {weight_kg}")
    except (ValueError, TypeError):
        errors.append(f"'weight_kg' must be numeric, got: {data['weight_kg']!r}")
        weight_kg = None

    # NORMALIZATION of activity_level
    activity_raw = str(data["activity_level"])
    activity, already_canonical = normalize_activity(activity_raw)
    if activity is None:
        activity = {"__llm_needed__": activity_raw}
        warnings.append(
            f"'activity_level' value '{activity_raw}' unrecognised — "
            f"will attempt LLM semantic normalization."
        )
    elif not already_canonical:
        normalizations.append(f"activity_level: '{activity_raw}' → '{activity}'")

    # NORMALIZATION of goal
    goal_raw = str(data["goal"])
    goal, already_canonical = normalize_goal(goal_raw)
    if goal is None:
        goal = {"__llm_needed__": goal_raw}
        warnings.append(
            f"'goal' value '{goal_raw}' unrecognised — "
            f"will attempt LLM semantic normalization."
        )
    elif not already_canonical:
        normalizations.append(f"goal: '{goal_raw}' → '{goal}'")

    # NORMALIZATION of diet_type
    diet_raw = str(data.get("diet_type", "balanced"))
    diet, already_canonical = normalize_diet(diet_raw)
    if diet is None:
        diet = "balanced"   # safe fallback for optional field
        warnings.append(
            f"'diet_type' value '{diet_raw}' unrecognised — defaulting to 'balanced'."
        )
    elif not already_canonical:
        normalizations.append(f"diet_type: '{diet_raw}' → '{diet}'")

    #medical_conditions
    med = data.get("medical_conditions", [])
    if not isinstance(med, list):
        if isinstance(med, str) and med.strip():
            med = [m.strip() for m in med.split(",")]
            normalizations.append(
                f"medical_conditions: converted string → list {med}"
            )
        else:
            med = []
            warnings.append("'medical_conditions' was not a list — reset to [].")


    if errors:
        return ValidationResult(False, None, errors, warnings, normalizations)

    # Build validated UserProfile
    profile = UserProfile(
        age=age,
        sex=sex,
        height_cm=height_cm,
        weight_kg=weight_kg,
        activity_level=activity,
        goal=goal,
        medical_conditions=med,
        diet_type=diet,
    )

    return ValidationResult(
        valid=True,
        profile=profile,
        errors=errors,
        warnings=warnings,
        normalizations=normalizations,
    )

Testing dataset

In [8]:
test_cases = [
    #  Clean input
    {
        "age": 22, "sex": "male", "height_cm": 175, "weight_kg": 75,
        "activity_level": "moderate", "goal": "muscle_gain",
        "medical_conditions": [], "diet_type": "balanced"
    },
    # normalization
    {
        "age": 30, "sex": "female", "height_cm": 160, "weight_kg": 60,
        "activity_level": "gym 3x/week", "goal": "lose weight",
        "diet_type": "plant based"
    },
    # Ambiguous goal
    {
        "age": 25, "sex": "male", "height_cm": 180, "weight_kg": 85,
        "activity_level": "active", "goal": "get shredded",
    },
    # Missing required field
    {
        "sex": "male", "height_cm": 170, "weight_kg": 70,
        "activity_level": "sedentary", "goal": "maintenance",
    },
    # Bad type for age
    {
        "age": "twenty two", "sex": "male", "height_cm": 175,
        "weight_kg": 75, "activity_level": "moderate", "goal": "muscle_gain",
    },
    # medical_conditions as comma string
    {
        "age": 40, "sex": "female", "height_cm": 165, "weight_kg": 70,
        "activity_level": "light", "goal": "maintenance",
        "medical_conditions": "diabetes, hypertension",
    },
]

In [113]:
labels = [
    "Clean canonical input",
    "Free-text normalization (rule-based)",
    "Ambiguous goal ('get shredded')",
    "Missing required field (age)",
    "Bad type (age = 'twenty two')",
    "medical_conditions as string",
]

for i, (tc, label) in enumerate(zip(test_cases, labels), 1):
    result = validate_input(tc)
    status = " VALID" if result.valid else " INVALID"
    print(f"{'='*60}")
    print(f"Test {i}: {label}")
    print(f"  Status         : {status}")
    if result.profile:
        print(f"  Profile        : {result.profile}")
    if result.errors:
        print(f"  Errors         : {result.errors}")
    if result.warnings:
        print(f"  Warnings       : {result.warnings}")
    if result.normalizations:
        print(f"  Normalizations : {result.normalizations}")
    print()

Test 1: Clean canonical input
  Status         :  VALID
  Profile        : UserProfile(age=22, sex='male', height_cm=175.0, weight_kg=75.0, activity_level='moderate', goal='muscle_gain', medical_conditions=[], diet_type='balanced')

Test 2: Free-text normalization (rule-based)
  Status         :  VALID
  Profile        : UserProfile(age=30, sex='female', height_cm=160.0, weight_kg=60.0, activity_level='moderate', goal='weight_loss', medical_conditions=[], diet_type='vegan')
  Normalizations : ["activity_level: 'gym 3x/week' → 'moderate'", "goal: 'lose weight' → 'weight_loss'", "diet_type: 'plant based' → 'vegan'"]

Test 3: Ambiguous goal ('get shredded')
  Status         :  VALID
  Profile        : UserProfile(age=25, sex='male', height_cm=180.0, weight_kg=85.0, activity_level='active', goal='weight_loss', medical_conditions=[], diet_type='balanced')
  Normalizations : ["goal: 'get shredded' → 'weight_loss'"]

Test 4: Missing required field (age)
  Status         :  INVALID
  Errors   

LLM fallback

In [10]:
def llm_semantic_normalization(profile: UserProfile) -> UserProfile:

    needs_llm = []
    if profile.activity_level == "__llm_needed__":
        needs_llm.append("activity_level")
    if profile.goal == "__llm_needed__":
        needs_llm.append("goal")

    if needs_llm:
        print(f"[LLM STUB] Fields pending normalization: {needs_llm}")
        print("  → Will be resolved in Step 3A using Llama 3.8")
    else:
        print("[LLM STUB] No LLM normalization needed — all fields resolved by rules.")

    return profile   # unchanged until Step 3 is implemented


#testing a case that need LLM
ambiguous_input = {
    "age": 28, "sex": "male", "height_cm": 178, "weight_kg": 80,
    "activity_level": "i go to the gym sometimes",
    "goal": "i want to look good for summer",
}

result = validate_input(ambiguous_input)
print("Validation result:")
print(f"  Valid    : {result.valid}")
print(f"  Warnings : {result.warnings}")
print()
if result.valid:
    llm_semantic_normalization(result.profile)

Validation result:
  Valid    : True
  Warnings : ["'activity_level' value 'i go to the gym sometimes' unrecognised — will attempt LLM semantic normalization.", "'goal' value 'i want to look good for summer' unrecognised — will attempt LLM semantic normalization."]

[LLM STUB] Fields pending normalization: ['activity_level', 'goal']
  → Will be resolved in Step 3A using Llama 3.8


In [38]:

GROQ_API_KEY = "<YOUR_GROQ_API_KEY>"

In [40]:
print(f"GROQ_API_KEY being used (first 10 chars): {GROQ_API_KEY[:10]}...")
print(f"Length of GROQ_API_KEY: {len(GROQ_API_KEY)}")


client = Groq(api_key=GROQ_API_KEY)

GROQ_API_KEY being used (first 10 chars): gsk_ehO9xx...
Length of GROQ_API_KEY: 56


In [13]:
!pip install groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 4.3 MB/s eta 0:00:00


In [55]:
from groq import Groq
import json

client = Groq(api_key=GROQ_API_KEY)

def llm_semantic_normalization(field_name, raw_value):
    prompt = f"""
You are a strict JSON classifier.

Your task:
Map the input into ONE allowed category.

Field: {field_name}
Input: "{raw_value}"

Allowed values:
- activity_level: sedentary, light, moderate, active
- goal: weight_loss, maintenance, muscle_gain

Rules:
- Output ONLY valid JSON
- Do NOT include explanations
- Do NOT include extra text
- Do NOT include markdown
- Output must be EXACTLY this format:

{{"value": "<one_of_allowed_values>"}}
"""

    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature=0
    )

    content = response.choices[0].message.content

    try:
        parsed = json.loads(content)
        return parsed["value"]
    except:
        print("LLM parsing error:", content)
        return None

In [56]:
def resolve_llm_fields(profile):

    if isinstance(profile.goal, dict):
        raw_value = profile.goal["__llm_needed__"]
        profile.goal = llm_semantic_normalization("goal", raw_value)

    if isinstance(profile.activity_level, dict):
        raw_value = profile.activity_level["__llm_needed__"]
        profile.activity_level = llm_semantic_normalization("activity_level", raw_value)

    return profile

# Test

In [114]:
test_input = {
    "age": 25,
    "sex": "male",
    "height_cm": 174,
    "weight_kg": 71,
    "activity_level": "i go gym daily",
    "goal": "be same"
}

result = validate_input(test_input)

profile = result.profile
profile = resolve_llm_fields(profile)

print(profile)

UserProfile(age=25, sex='male', height_cm=174.0, weight_kg=71.0, activity_level='active', goal='maintenance', medical_conditions=[], diet_type='balanced')


Safety layer

In [64]:
bmi = profile.weight_kg / ((profile.height_cm / 100) ** 2)

In [65]:
def safety_check(profile):
    warnings = []

    height_m = profile.height_cm / 100
    bmi = profile.weight_kg / (height_m ** 2)

    if profile.age < 18:
        return {"status": "needs_review", "reason": "Underage"}

    if bmi < 18.5:
        return {"status": "needs_review", "reason": "Underweight"}

    if bmi > 35:
        return {"status": "needs_review", "reason": "Severely obese"}

    if profile.medical_conditions:
        return {"status": "needs_review", "reason": "Medical condition present"}

    # warnings
    if bmi < 20:
        warnings.append("Low BMI")

    if bmi > 30:
        warnings.append("High BMI")

    return {
        "status": "safe",
        "bmi": round(bmi, 2),
        "warnings": warnings
    }

In [75]:
profile = result.profile
profile = resolve_llm_fields(profile)

safety = safety_check(profile)

print("Profile:", profile)
print("Safety:", safety)

Profile: UserProfile(age=25, sex='male', height_cm=180.0, weight_kg=225.0, activity_level='active', goal='muscle_gain', medical_conditions=[], diet_type='balanced')
Safety: {'status': 'needs_review', 'reason': 'Severely obese'}


Calculations

In [78]:
def calculate_bmr(profile):
    if profile.sex == "male":
        return 10 * profile.weight_kg + 6.25 * profile.height_cm - 5 * profile.age + 5
    else:
        return 10 * profile.weight_kg + 6.25 * profile.height_cm - 5 * profile.age - 161

In [79]:
def calculate_tdee(bmr, activity_level):

    factors = {
        "sedentary": 1.2,
        "light": 1.375,
        "moderate": 1.55,
        "active": 1.725
    }

    return bmr * factors[activity_level]

In [80]:
def apply_goal(tdee, goal):
    if goal == "weight_loss":
        return tdee - 400
    elif goal == "muscle_gain":
        return tdee + 300
    return tdee

In [81]:
def calculate_macros(calories, profile):

    # protein
    if profile.goal == "weight_loss":
        protein = profile.weight_kg * 1.8
    elif profile.goal == "muscle_gain":
        protein = profile.weight_kg * 2.0
    else:
        protein = profile.weight_kg * 1.4

    # fat = 25% calories
    fat_cal = calories * 0.25
    fat = fat_cal / 9

    # car = cal - prot - fat
    protein_cal = protein * 4
    carb_cal = calories - (protein_cal + fat_cal)
    carbs = carb_cal / 4

    return round(protein), round(fat), round(carbs)

In [82]:
def calculate_targets(profile):

    bmr = calculate_bmr(profile)
    tdee = calculate_tdee(bmr, profile.activity_level)
    calories = apply_goal(tdee, profile.goal)

    # safety constraint
    if calories < 1000:
        calories = 1000

    protein, fat, carbs = calculate_macros(calories, profile)

    return {
        "bmr": round(bmr),
        "tdee": round(tdee),
        "daily_calories": round(calories),
        "macros": {
            "protein_g": protein,
            "fat_g": fat,
            "carbs_g": carbs
        }
    }

In [97]:
profile = resolve_llm_fields(result.profile)
safety = safety_check(profile)

if safety["status"] == "safe":
    targets = calculate_targets(profile)
    print(targets)
else:
    print("Blocked:", safety)

{'bmr': 1668, 'tdee': 2001, 'daily_calories': 2301, 'macros': {'protein_g': 140, 'fat_g': 64, 'carbs_g': 291}}


In [104]:
def generate_explanation(profile, targets):

    prompt = f"""
You are a nutrition assistant.

Explain the following nutrition targets in a short, clear way.

User:
- Age: {profile.age}
- Sex: {profile.sex}
- Weight: {profile.weight_kg} kg
- Height: {profile.height_cm} cm
- Activity: {profile.activity_level}
- Goal: {profile.goal}

Targets:
- Calories: {targets["daily_calories"]}
- Protein: {targets["macros"]["protein_g"]} g
- Fat: {targets["macros"]["fat_g"]} g
- Carbs: {targets["macros"]["carbs_g"]} g

Rules:
- 1–2 sentences only
- No medical advice
- No extra formatting
"""

    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.3
    )

    return response.choices[0].message.content.strip()


In [110]:
targets = calculate_targets(profile)
explanation = generate_explanation(profile, targets)

print(explanation)

To achieve weight loss, this individual should consume approximately 2494 calories per day, which is a deficit from their daily needs due to their active lifestyle. 

Their macronutrient targets are: 
- 128 grams of protein to support muscle health and satisfaction
- 69 grams of fat for energy and hormone regulation
- 340 grams of carbohydrates for energy and fiber intake.


full pipeline

In [115]:
def run_target_agent(raw_input):


    result = validate_input(raw_input)
    if not result.valid:
        return {"status": "error", "errors": result.errors}

    profile = result.profile


    profile = resolve_llm_fields(profile)


    safety = safety_check(profile)
    if safety["status"] != "safe":
        return {"status": "blocked", "reason": safety}


    targets = calculate_targets(profile)


    explanation = generate_explanation(profile, targets)

    return {
        "status": "success",
        "profile": profile,
        "targets": targets,
        "explanation": explanation
    }

In [116]:
output = run_target_agent(test_input)
print(output)

{'status': 'success', 'profile': UserProfile(age=25, sex='male', height_cm=174.0, weight_kg=71.0, activity_level='active', goal='maintenance', medical_conditions=[], diet_type='balanced'), 'targets': {'bmr': 1678, 'tdee': 2894, 'daily_calories': 2894, 'macros': {'protein_g': 99, 'fat_g': 80, 'carbs_g': 443}}, 'explanation': 'To maintain your weight, your daily targets are: \n- Calories: 2894 to support your active lifestyle.\n- Protein: 99 grams to help build and repair muscles.\n- Fat: 80 grams, which is a moderate amount for an active person.\n- Carbs: 443 grams, providing energy for your daily activities.'}
